In [1]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\train_df_prepared.parquet")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\valid_df_prepared.parquet")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\test_df_prepared.parquet")

# Chuyển đổi timestamp sang số (giây)
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

# Sắp xếp tĩnh trực tiếp theo thời gian
train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

print("Dữ liệu đã được load và sắp xếp theo thời gian thành công!")


Dữ liệu đã được load và sắp xếp theo thời gian thành công!


In [2]:
import numpy as np
import pandas as pd
import torch

def create_graph_snapshots(df, feature_cols, label_col='label', window_size=2000, overlap=1500):
    """
    Cắt DataFrame thành các Graph Snapshots dựa trên cơ chế Sliding Window.
    
    Args:
        df (pd.DataFrame): DataFrame đã được sort theo timestamp.
        feature_cols (list): Danh sách các cột đặc trưng hành vi (KHÔNG chứa IP, Port, Timestamp).
        label_col (str): Tên cột chứa nhãn tấn công.
        window_size (int): Số lượng luồng (flow) trong một snapshot.
        overlap (int): Số lượng luồng giữ lại từ snapshot trước đó.
        
    Returns:
        list: Danh sách các dictionary, mỗi dict là một snapshot chứa 'x', 'y', và 'meta'.
    """
    snapshots = []
    
    # Tính bước trượt (stride)
    stride = window_size - overlap
    if stride <= 0:
        raise ValueError("Overlap phải nhỏ hơn window_size!")
        
    num_flows = len(df)
    
    # Ép kiểu dữ liệu về Numpy array trước để tăng tốc độ cắt (slicing)
    # Tốc độ cắt trên Numpy array nhanh hơn pandas.iloc rất nhiều
    X_all = df[feature_cols].values.astype(np.float32)
    Y_all = df[label_col].values.astype(np.int64)
    
    # Trích xuất Metadata để phục vụ việc nối cạnh (Edge Construction) ở bước sau
    dst_ips = df['dst_ip'].values
    dst_ports = df['dst_port'].values
    timestamps = df['timestamp_num'].values
    
    print(f"Bắt đầu tạo snapshot: Tổng số {num_flows} flows, Window={window_size}, Stride={stride}")
    
    # Quét qua toàn bộ dữ liệu với bước nhảy là stride
    for start_idx in range(0, num_flows - window_size + 1, stride):
        end_idx = start_idx + window_size
        
        # 1. Trích xuất Node Features và Labels cho Snapshot hiện tại
        x_snapshot = X_all[start_idx:end_idx]
        y_snapshot = Y_all[start_idx:end_idx]
        
        # 2. Trích xuất Metadata tương ứng
        meta_snapshot = {
            'dst_ip': dst_ips[start_idx:end_idx],
            'dst_port': dst_ports[start_idx:end_idx],
            'timestamp': timestamps[start_idx:end_idx],
            'global_indices': np.arange(start_idx, end_idx) # Lưu lại index gốc để track
        }
        
        # 3. Đóng gói vào dictionary (Chuyển luôn sang PyTorch Tensor để tiện dùng sau này)
        snapshot_data = {
            'x': torch.tensor(x_snapshot),
            'y': torch.tensor(y_snapshot),
            'meta': meta_snapshot
        }
        
        snapshots.append(snapshot_data)
        
    print(f"Đã tạo thành công {len(snapshots)} snapshots!")
    return snapshots

In [3]:
# Ví dụ thiết lập tham số
WINDOW_SIZE = 4000
OVERLAP = 2500
feature_cols = [col for col in train_df.columns if col not in ["flow_id",'timestamp', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'label', 'timestamp_num', "src_port", "dst_port"]]
print(f"Các cột đặc trưng được sử dụng: {len(feature_cols)} cột")
# Tạo snapshots cho tập Train
train_snapshots = create_graph_snapshots(
    df=train_df, 
    feature_cols=feature_cols, 
    label_col='label', 
    window_size=WINDOW_SIZE, 
    overlap=OVERLAP
)

# Tạo snapshots cho tập Valid và Test
valid_snapshots = create_graph_snapshots(
    df=valid_df, 
    feature_cols=feature_cols, 
    label_col='label', 
    window_size=WINDOW_SIZE, 
    overlap=OVERLAP
)

test_snapshots = create_graph_snapshots(
    df=test_df, 
    feature_cols=feature_cols, 
    label_col='label', 
    window_size=WINDOW_SIZE, 
    overlap=OVERLAP
)

Các cột đặc trưng được sử dụng: 323 cột
Bắt đầu tạo snapshot: Tổng số 3294945 flows, Window=4000, Stride=1500
Đã tạo thành công 2194 snapshots!
Bắt đầu tạo snapshot: Tổng số 4393262 flows, Window=4000, Stride=1500
Đã tạo thành công 2927 snapshots!
Bắt đầu tạo snapshot: Tổng số 3294963 flows, Window=4000, Stride=1500
Đã tạo thành công 2194 snapshots!


In [4]:
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from tqdm import tqdm  # Thư viện tạo thanh tiến trình chuyên nghiệp
from collections import defaultdict # Thư viện để gom nhóm dữ liệu

def build_graph_from_snapshot(snapshot, k=10, alpha=0.7, beta=0.3, lambda_decay=3.0):
    """
    Xử lý một snapshot đơn lẻ (Đã được tối ưu hóa: gom nhóm node giúp giảm từ O(N^2) xuống gần O(N)).
    """
    x = snapshot['x']
    y = snapshot['y']
    meta = snapshot['meta']
    num_nodes = x.shape[0]

    timestamps = meta['timestamp']
    edge_index_list = []

    # 1. TỐI ƯU HÓA: Gom nhóm các node có chung target (IP Đích, Port Đích)
    target_groups = defaultdict(list)
    # Dùng tuple (ip, port) làm key xử lý hash liên tục vô cùng nhanh
    for idx, (ip, port) in enumerate(zip(meta['dst_ip'], meta['dst_port'])):
        target_groups[(ip, port)].append(idx)

    # 2. Xây dựng cạnh trong THUỘC TỪNG NHÓM (Chỉ tính những node có khả năng liên kết)
    for target_key, indices_list in target_groups.items():
        # Nếu nhóm có 1 node, nó đứng cô lập, khỏi cần tính cạnh
        if len(indices_list) < 2:
            continue
            
        indices = np.array(indices_list)
        group_timestamps = timestamps[indices]
        
        # Chỉ quét mảng nhỏ của riêng nhóm này thay vì toàn bộ num_nodes
        for local_idx, global_i in enumerate(indices):
            # Tạo mask để bỏ qua mốc index của chính node đó
            mask = np.ones(len(indices), dtype=bool)
            mask[local_idx] = False
            
            other_indices = indices[mask]
            other_timestamps = group_timestamps[mask]
            
            # Tính độ chênh lệch thời gian
            time_diffs = np.abs(other_timestamps - group_timestamps[local_idx])
            k_actual = min(k, len(time_diffs))
            
            # Sử dụng argpartition tăng tốc độ hơn argsort (chỉ cần lấy K nhỏ nhất, ko cần sort full mảng)
            if k_actual < len(time_diffs):
                top_k_local_indices = np.argpartition(time_diffs, k_actual - 1)[:k_actual]
            else:
                top_k_local_indices = np.arange(len(time_diffs))
                
            top_k_global_indices = other_indices[top_k_local_indices]
            
            # Kết nối các cặp [Source, Dest]
            for global_j in top_k_global_indices:
                edge_index_list.append([global_i, global_j])

    if not edge_index_list:
        return Data(x=x, edge_index=torch.empty((2, 0), dtype=torch.long), 
                    edge_attr=torch.empty((0,), dtype=torch.float32), y=y)

    edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()

    # Tính Cosine Similarity
    src_nodes_features = x[edge_index[0]]
    dst_nodes_features = x[edge_index[1]]
    cos_sim = F.cosine_similarity(src_nodes_features, dst_nodes_features, dim=1, eps=1e-8)
    cos_sim = (cos_sim + 1.0) / 2.0 

    # Tính Time Decay với chuẩn hóa Min-Max cục bộ
    src_times = torch.tensor(timestamps[edge_index[0].numpy()], dtype=torch.float32)
    dst_times = torch.tensor(timestamps[edge_index[1].numpy()], dtype=torch.float32)
    time_diffs_tensor = torch.abs(src_times - dst_times)
    
    max_diff = time_diffs_tensor.max()
    if max_diff > 0:
        time_diffs_norm = time_diffs_tensor / max_diff
    else:
        time_diffs_norm = time_diffs_tensor
        
    time_decay = torch.exp(-lambda_decay * time_diffs_norm)
    edge_weight = alpha * cos_sim + beta * time_decay

    return Data(x=x, edge_index=edge_index, edge_attr=edge_weight, y=y)


def convert_all_snapshots_to_graphs(snapshots_list, dataset_name="Train Set", k=10, alpha=0.7, beta=0.3, lambda_decay=3.0):
    """
    Hàm wrapper bọc tqdm bên ngoài để track tiến trình tạo đồ thị cho TOÀN BỘ danh sách snapshots.
    """
    graph_objects = []
    
    # tqdm sẽ tự động tính toán % tiến độ, số snapshot/s và thời gian còn lại
    progress_bar = tqdm(
        snapshots_list, 
        desc=f"Building Graphs ({dataset_name})", 
        unit="snapshot",
        ncols=100 # Độ rộng của thanh tiến trình hiển thị trên console
    )
    
    for snap in progress_bar:
        graph_data = build_graph_from_snapshot(
            snapshot=snap, 
            k=k, 
            alpha=alpha, 
            beta=beta, 
            lambda_decay=lambda_decay
        )
        graph_objects.append(graph_data)
        
    return graph_objects

In [5]:
import gc

# Thực hiện tạo đồ thị và track tiến trình cho tập Train
train_graphs = convert_all_snapshots_to_graphs(train_snapshots, dataset_name="Train Set")
# Giải phóng snapshots ngay sau khi tạo đồ thị xong
del train_snapshots
gc.collect()

# Thực hiện tạo đồ thị và track tiến trình cho tập Valid
valid_graphs = convert_all_snapshots_to_graphs(valid_snapshots, dataset_name="Valid Set")
del valid_snapshots
gc.collect()

# Thực hiện tạo đồ thị và track tiến trình cho tập Test
test_graphs = convert_all_snapshots_to_graphs(test_snapshots, dataset_name="Test Set")
del test_snapshots
gc.collect()

# --- TỐI ƯU HÓA RAM CPU CỰC MẠNH ---
# Sau khi đã có DataLoader và graph objects, ta KHÔNG CÒN CẦN các DataFrame Pandas ban đầu nữa.
# Chúng chiếm dung lượng khổng lồ trên RAM hệ thống (System RAM), cần giải phóng ngay!
try:
    del train_df, valid_df, test_df
    gc.collect()
    print("Đã giải phóng hoàn toàn các DataFrame Pandas khỏi RAM hệ thống!")
except NameError:
    pass


Building Graphs (Test Set): 100%|█████████████████████████| 2194/2194 [04:17<00:00,  8.52snapshot/s]

Đã giải phóng hoàn toàn các DataFrame Pandas khỏi RAM hệ thống!


In [6]:
from torch_geometric.loader import DataLoader

# Thiết lập Batch Size (Số lượng snapshot đưa vào GPU trong 1 bước)
# VRAM GPU đang bị tràn (CUDA Out of Memory) do GAT Model tốn nhiều bộ nhớ để tính toán Multi-head Attention
# CẦN GIẢM MẠNH BATCH SIZE xuống (VD: từ 32 xuống 8 hoặc 16)
BATCH_SIZE = 8 

# Khởi tạo DataLoader cho cả 3 tập
# Lưu ý: Tập Train cần shuffle=True để mô hình học khách quan, Valid/Test giữ nguyên thứ tự
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_graphs, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

print(f"Đã chuẩn bị xong DataLoader:")
print(f" - Train Batches: {len(train_loader)}")
print(f" - Valid Batches: {len(valid_loader)}")
print(f" - Test Batches: {len(test_loader)}")

Đã chuẩn bị xong DataLoader:
 - Train Batches: 275
 - Valid Batches: 366
 - Test Batches: 275


In [7]:
import torch
import numpy as np

class EarlyStopping:
    """
    Trình quản lý dừng sớm (Early Stopping) để theo dõi Macro F1 trên tập Validation.
    """
    def __init__(self, patience=8, path='best_baseline_gcn.pt'):
        self.patience = patience
        self.path = path
        self.counter = 0
        self.best_f1 = -float('inf')
        self.early_stop = False

    def __call__(self, val_f1, model):
        # Nếu F1 cải thiện (lớn hơn điểm tốt nhất trước đó)
        if val_f1 > self.best_f1:
            self.best_f1 = val_f1
            self.save_checkpoint(model)
            self.counter = 0 # Reset bộ đếm về 0
        else:
            # Nếu không cải thiện, tăng bộ đếm thêm 1
            self.counter += 1
            print(f' -> [EarlyStopping] Bộ đếm tăng: {self.counter} / {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, model):
        '''Lưu mô hình khi điểm Validation F1 tăng'''
        torch.save(model.state_dict(), self.path)
        print(f' -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.')

In [8]:
# Chuyển mô hình lên GPU (nếu có)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Chạy đoạn code sinh class_weights của bạn
# LƯU Ý: all_train_labels phải là nhãn của TẬP TRAIN (Không gộp Valid/Test vào để tránh Data Leakage)
import sklearn.utils.class_weight as class_weight

# VÌ CHÚNG TA ĐÃ XÓA train_df ĐỂ TIẾT KIỆM RAM (Nhằm tránh lỗi Out of Memory), 
# ta sẽ rút danh sách nhãn trực tiếp từ các đồ thị train_graphs
all_train_labels = np.concatenate([g.y.numpy() for g in train_graphs])
unique_classes = np.unique(all_train_labels) # Lấy các class có trong tập train

class_weights = class_weight.compute_class_weight(
    class_weight='balanced', 
    classes=unique_classes, 
    y=all_train_labels
)

smoothed_weight = np.sqrt(class_weights)
class_weight = np.clip(smoothed_weight, a_min=0.1, a_max=10.0)

# Đưa lên thiết bị (GPU/CPU)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device) 

# ==========================================
# 2. Áp dụng vào Hàm Mất Mát (Loss Function)
# ==========================================
# Chèn class_weights_tensor vào tham số 'weight' của NLLLoss
criterion = torch.nn.NLLLoss(weight=class_weights_tensor)

print("Đã tích hợp Class Weights vào hàm Loss thành công!")
print(f"Trọng số các class: {class_weights}")

Đã tích hợp Class Weights vào hàm Loss thành công!
Trọng số các class: [1.02908068e+00 4.12852956e-01 2.74915467e+00 4.58222613e-01
 9.19127273e-01 1.08185404e+00 3.67137849e-01 3.66584809e+00
 1.60935686e+00 1.25478982e+02 4.97111136e+02 9.73813775e+00
 7.59260714e-01]


In [9]:
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score
import gc # Thư viện dọn rác hệ thống

def train_epoch(epoch, EPOCHS):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    # Thanh tiến trình cho tập Train
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch:03d}/{EPOCHS} [Train]", leave=False, ncols=100)
    
    for batch in progress_bar:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = criterion(out, batch.y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch.num_graphs
        pred = out.argmax(dim=1)
        
        # detach() ngắt kết nối với đồ thị tính toán GPU trước khi đẩy về CPU
        all_preds.extend(pred.detach().cpu().numpy())
        all_labels.extend(batch.y.detach().cpu().numpy())
        
        # Cập nhật số loss liên tục trên thanh tiến trình
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
        
        # ----------------------------------------------------
        # TỐI ƯU VRAM: Giải phóng vùng nhớ GPU ngay sau mỗi bước
        # ----------------------------------------------------
        del batch, out, loss, pred
        
    # Xóa bộ nhớ đệm (Cache) của GPU sau khi hết 1 Epoch để tránh rác cộng dồn
    torch.cuda.empty_cache() 
    gc.collect()
        
    loss_avg = total_loss / len(train_loader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return loss_avg, macro_f1

@torch.no_grad()
def valid_epoch(epoch, EPOCHS):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    # Thanh tiến trình cho tập Valid
    progress_bar = tqdm(valid_loader, desc=f"Epoch {epoch:03d}/{EPOCHS} [Valid]", leave=False, ncols=100)
    
    for batch in progress_bar:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = criterion(out, batch.y)
        
        total_loss += loss.item() * batch.num_graphs
        pred = out.argmax(dim=1)
        
        all_preds.extend(pred.detach().cpu().numpy())
        all_labels.extend(batch.y.detach().cpu().numpy())
        
        # TỐI ƯU VRAM GPU
        del batch, out, loss, pred
        
    torch.cuda.empty_cache() 
    gc.collect()
        
    loss_avg = total_loss / len(valid_loader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return loss_avg, macro_f1

In [10]:
from sklearn.metrics import f1_score
from tqdm import tqdm
import torch

@torch.no_grad()
def test_benchmark_new_nodes(loader, window_size=2000, stride=500):
    model.eval() # Tắt tính toán gradient, tắt Dropout
    
    all_preds = []
    all_labels = []
    
    overlap = window_size - stride
    curr_snap_idx = 0
    
    print("Đang chạy Benchmark trên tập Test (Chiến lược: Chỉ đánh giá luồng Mới)...")
    progress_bar = tqdm(loader, desc="Test Benchmark", ncols=100)
    
    for batch in progress_bar:
        batch = batch.to(device)
        
        # Chạy forward pass qua mô hình
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        
        # Dự đoán class có xác suất cao nhất ngay lập tức
        preds = out.argmax(dim=1)
        
        # Bóc tách từng đồ thị trong batch để cắt lấy các Node mới
        for g_idx in range(batch.num_graphs):
            # Tạo mask để lấy đúng các node thuộc về đồ thị thứ g_idx
            node_mask = (batch.batch == g_idx)
            
            g_preds = preds[node_mask]
            g_labels = batch.y[node_mask]
            g_num_nodes = g_preds.size(0)
            
            # Logic cắt node mới
            if curr_snap_idx == 0:
                # Snapshot đầu tiên: Đánh giá toàn bộ
                start_eval_idx = 0
            else:
                # Các Snapshot sau: Bỏ qua phần overlap, chỉ lấy phần mới
                start_eval_idx = overlap
                
            # Đề phòng trường hợp snapshot cuối cùng bị thiếu node
            if start_eval_idx < g_num_nodes:
                new_preds = g_preds[start_eval_idx:g_num_nodes]
                new_labels = g_labels[start_eval_idx:g_num_nodes]
                
                # Nạp vào danh sách tổng
                all_preds.extend(new_preds.cpu().numpy())
                all_labels.extend(new_labels.cpu().numpy())
            
            curr_snap_idx += 1
            
    # Tính F1 Score cuối cùng
    final_macro_f1 = f1_score(all_labels, all_preds, average='macro')
    
    print(f"Tổng số flow thực tế được đánh giá: {len(all_labels)}")
    print(f"=====================================")
    print(f"🏆 CHÍNH THỨC - TEST BENCHMARK MACRO F1: {final_macro_f1:.4f}")
    print(f"=====================================")

    # in ra classification report chi tiết
    from sklearn.metrics import classification_report
    print("\n📊 Classification Report Chi Tiết:")
    print(classification_report(all_labels, all_preds, digits=4))
    
    return final_macro_f1, all_preds, all_labels

Thử bằng model GAT

In [11]:
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import GATConv

class AdvancedGAT(torch.nn.Module):
    def __init__(self, num_node_features, num_classes=13):
        super(AdvancedGAT, self).__init__()
        
        # 1. Lớp đầu vào (Giữ nguyên để nén feature)
        self.lin_in = Linear(num_node_features, 32)
        
        # 2. Các lớp GAT với Multi-head Attention
        # Lớp 1: Input 32 -> 4 heads x 16 = 64 chiều (Tương đương GCNConv 64)
        self.gat1 = GATConv(
            in_channels=32, out_channels=16, heads=4, concat=True, edge_dim=1
        )
        
        # Lớp 2: Input 64 -> 4 heads x 32 = 128 chiều (Tương đương GCNConv 128)
        self.gat2 = GATConv(
            in_channels=64, out_channels=32, heads=4, concat=True, edge_dim=1
        )
        
        # Lớp 3: Input 128 -> 4 heads x 64. 
        # CHÚ Ý: Ở lớp GNN cuối cùng, ta dùng concat=False để tính trung bình các heads, 
        # gom lại thành 64 chiều chuẩn bị cho lớp Linear.
        self.gat3 = GATConv(
            in_channels=128, out_channels=64, heads=4, concat=False, edge_dim=1
        )
        
        # 3. Lớp phân loại đầu ra
        self.lin_out = Linear(64, num_classes)

    def forward(self, x, edge_index, edge_attr):
        # TIỂU XẢO KỸ THUẬT QUAN TRỌNG:
        # GATConv yêu cầu edge_attr phải là ma trận 2 chiều [Số cạnh, Số feature cạnh].
        # Trọng số Cosine + Time Decay của ta đang là mảng 1 chiều [E], ta cần thêm 1 trục (unsqueeze).
        if edge_attr.dim() == 1:
            edge_attr = edge_attr.unsqueeze(-1)
            
        x = self.lin_in(x)
        x = F.relu(x)
        
        # Lớp GAT 1 (Dùng hàm ELU thay vì ReLU vì các paper về GAT chứng minh ELU mượt mà hơn cho Attention)
        x = self.gat1(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x) 
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp GAT 2
        x = self.gat2(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp GAT 3
        x = self.gat3(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp Output
        x = self.lin_out(x)
        
        return F.log_softmax(x, dim=1)

In [12]:
# 1. Khởi tạo GAT Model
model = AdvancedGAT(num_node_features=len(feature_cols), num_classes=13) # Lấy động features từ mảng cột DataFrame
model = model.to(device)

# 2. Khởi tạo lại Optimizer (Bắt buộc phải tạo lại để nó theo dõi trọng số của GAT)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

# (Hàm criterion NLLLoss với class_weights của bạn vẫn giữ nguyên không đổi)

# 3. Reset Early Stopping (Nên đổi tên file lưu để không ghi đè mất file GCN cũ)
early_stopping = EarlyStopping(patience=8, path='best_advanced_gat.pt')

print(model)

AdvancedGAT(
  (lin_in): Linear(in_features=323, out_features=32, bias=True)
  (gat1): GATConv(32, 16, heads=4)
  (gat2): GATConv(64, 32, heads=4)
  (gat3): GATConv(128, 64, heads=4)
  (lin_out): Linear(in_features=64, out_features=13, bias=True)
)


In [13]:
EPOCHS = 50

# Khởi tạo bộ quản lý Early Stopping với patience = 8
early_stopping = EarlyStopping(patience=8, path='best_advanced_gat.pt')

print("Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...")
for epoch in range(1, EPOCHS + 1):
    # 1. Chạy huấn luyện và lấy kết quả
    train_loss, train_f1 = train_epoch(epoch, EPOCHS)
    
    # 2. Chạy validation đánh giá
    valid_loss, valid_f1 = valid_epoch(epoch, EPOCHS)
    
    # In kết quả của epoch hiện tại
    print(f"Epoch: {epoch:03d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Macro F1: {train_f1:.4f} | "
          f"Valid Loss: {valid_loss:.4f} - Macro F1: {valid_f1:.4f}")
    
    # 3. Kiểm tra điều kiện dừng sớm
    early_stopping(valid_f1, model)
      
    if early_stopping.early_stop:
        print(f"\n🛑 [DỪNG SỚM] Mô hình không cải thiện sau {early_stopping.patience} Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch {epoch}!")
        break

Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...


Epoch: 001/50 | Train Loss: 0.6487 - Macro F1: 0.6255 | Valid Loss: 0.2112 - Macro F1: 0.7135
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 002/50 | Train Loss: 0.2521 - Macro F1: 0.7545 | Valid Loss: 0.1376 - Macro F1: 0.7281
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 003/50 | Train Loss: 0.1724 - Macro F1: 0.8003 | Valid Loss: 0.0878 - Macro F1: 0.7952
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 004/50 | Train Loss: 0.1372 - Macro F1: 0.8284 | Valid Loss: 0.0724 - Macro F1: 0.8773
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


...

Epoch: 034/50 | Train Loss: 0.0071 - Macro F1: 0.9891 | Valid Loss: 0.0123 - Macro F1: 0.9976
 -> [EarlyStopping] Bộ đếm tăng: 4 / 8


Epoch: 035/50 | Train Loss: 0.0107 - Macro F1: 0.9885 | Valid Loss: 0.1379 - Macro F1: 0.7941
 -> [EarlyStopping] Bộ đếm tăng: 5 / 8


Epoch: 036/50 | Train Loss: 0.1099 - Macro F1: 0.8705 | Valid Loss: 0.0485 - Macro F1: 0.8527
 -> [EarlyStopping] Bộ đếm tăng: 6 / 8


Epoch: 037/50 | Train Loss: 0.0823 - Macro F1: 0.8626 | Valid Loss: 0.0320 - Macro F1: 0.8807
 -> [EarlyStopping] Bộ đếm tăng: 7 / 8


Epoch: 038/50 | Train Loss: 0.0567 - Macro F1: 0.8866 | Valid Loss: 0.0507 - Macro F1: 0.9552
 -> [EarlyStopping] Bộ đếm tăng: 8 / 8

🛑 [DỪNG SỚM] Mô hình không cải thiện sau 8 Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch 38!


In [14]:
# 1. Nạp lại trọng số tối ưu nhất đã được Early Stopping lưu lại trước đó
model.load_state_dict(torch.load('best_advanced_gat.pt'))
model = model.to(device)
print("Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!")

# 2. Chạy Benchmark chính thức trên tập Test (Chỉ lấy node mới, không tính trùng overlap)
test_f1, test_preds, test_labels = test_benchmark_new_nodes(
    loader=test_loader, 
    window_size=2000, 
    stride=500
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_17652\610466301.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_advanced_gat.pt'))


Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!
Đang chạy Benchmark trên tập Test (Chiến lược: Chỉ đánh giá luồng Mới)...


Test Benchmark:   0%|                                                       | 0/275 [00:00<?, ?it/s]

Test Benchmark: 100%|█████████████████████████████████████████████| 275/275 [00:24<00:00, 11.40it/s]


Tổng số flow thực tế được đánh giá: 5486500
🏆 CHÍNH THỨC - TEST BENCHMARK MACRO F1: 0.9083

📊 Classification Report Chi Tiết:
              precision    recall  f1-score   support

           0     0.9974    0.9969    0.9972    410000
           1     0.9973    0.9990    0.9981   1021936
           2     0.9924    0.7823    0.8749    153398
           3     1.0000    0.9985    0.9993    920554
           4     0.9978    0.9964    0.9971    458988
           5     0.9972    0.9974    0.9973    390000
           6     1.0000    1.0000    1.0000   1149290
           7     0.9999    0.7300    0.8439    115104
           8     1.0000    1.0000    1.0000    262198
           9     0.3671    1.0000    0.5370      3424
          10     0.9661    0.9846    0.9752      1360
          11     0.4193    0.9804    0.5874     43362
          12     1.0000    1.0000    1.0000    556886

    accuracy                         0.9869   5486500
   macro avg     0.9026    0.9589    0.9083   5486500
weighted

In [16]:
EPOCHS = 50
model = AdvancedGAT(num_node_features=len(feature_cols), num_classes=13) # Lấy động features từ mảng cột DataFrame
model = model.to(device)

# 2. Khởi tạo lại Optimizer (Bắt buộc phải tạo lại để nó theo dõi trọng số của GAT)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)
# (Hàm criterion NLLLoss với class_weights của bạn vẫn giữ nguyên không đổi)

# 3. Reset Early Stopping (Nên đổi tên file lưu để không ghi đè mất file GCN cũ)
early_stopping = EarlyStopping(patience=10, path='best_advanced_gat.pt')


print("Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...")
for epoch in range(1, EPOCHS + 1):
    # 1. Chạy huấn luyện và lấy kết quả
    train_loss, train_f1 = train_epoch(epoch, EPOCHS)
    
    # 2. Chạy validation đánh giá
    valid_loss, valid_f1 = valid_epoch(epoch, EPOCHS)
    
    # In kết quả của epoch hiện tại
    print(f"Epoch: {epoch:03d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Macro F1: {train_f1:.4f} | "
          f"Valid Loss: {valid_loss:.4f} - Macro F1: {valid_f1:.4f}")
    
    # 3. Kiểm tra điều kiện dừng sớm
    early_stopping(valid_f1, model)
    scheduler.step(valid_f1) # Điều chỉnh learning rate dựa trên Macro F1 của Validation
    if early_stopping.early_stop:
        print(f"\n🛑 [DỪNG SỚM] Mô hình không cải thiện sau {early_stopping.patience} Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch {epoch}!")
        break
# 1. Nạp lại trọng số tối ưu nhất đã được Early Stopping lưu lại trước đó
model.load_state_dict(torch.load('best_advanced_gat.pt'))
model = model.to(device)
print("Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!")

# 2. Chạy Benchmark chính thức trên tập Test (Chỉ lấy node mới, không tính trùng overlap)
test_f1, test_preds, test_labels = test_benchmark_new_nodes(
    loader=test_loader, 
    window_size=2000, 
    stride=500
)

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...


Epoch: 001/50 | Train Loss: 0.6589 - Macro F1: 0.6132 | Valid Loss: 0.1676 - Macro F1: 0.7345
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 002/50 | Train Loss: 0.2205 - Macro F1: 0.7603 | Valid Loss: 0.1659 - Macro F1: 0.7428
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 003/50 | Train Loss: 0.1150 - Macro F1: 0.8357 | Valid Loss: 0.1680 - Macro F1: 0.8423
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


...

Epoch: 047/50 | Train Loss: 0.0064 - Macro F1: 0.9935 | Valid Loss: 0.0084 - Macro F1: 0.9971
 -> [EarlyStopping] Bộ đếm tăng: 9 / 10


C:\Users\Admin\AppData\Local\Temp\ipykernel_17652\3992018802.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_advanced_gat.pt'))


Epoch: 048/50 | Train Loss: 0.0062 - Macro F1: 0.9938 | Valid Loss: 0.0087 - Macro F1: 0.9970
 -> [EarlyStopping] Bộ đếm tăng: 10 / 10

🛑 [DỪNG SỚM] Mô hình không cải thiện sau 10 Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch 48!
Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!
Đang chạy Benchmark trên tập Test (Chiến lược: Chỉ đánh giá luồng Mới)...


Test Benchmark: 100%|█████████████████████████████████████████████| 275/275 [00:22<00:00, 11.98it/s]


Tổng số flow thực tế được đánh giá: 5486500
🏆 CHÍNH THỨC - TEST BENCHMARK MACRO F1: 0.9303

📊 Classification Report Chi Tiết:
              precision    recall  f1-score   support

           0     0.9997    0.9722    0.9857    410000
           1     0.9863    0.9998    0.9930   1021936
           2     0.9667    0.7738    0.8595    153398
           3     1.0000    0.9987    0.9993    920554
           4     0.9971    0.9841    0.9905    458988
           5     0.9996    0.9966    0.9981    390000
           6     0.9999    1.0000    1.0000   1149290
           7     1.0000    0.8675    0.9291    115104
           8     0.9802    1.0000    0.9900    262198
           9     0.6295    0.9603    0.7605      3424
          10     0.8997    0.9890    0.9422      1360
          11     0.4824    0.9742    0.6452     43362
          12     1.0000    1.0000    1.0000    556886

    accuracy                         0.9868   5486500
   macro avg     0.9185    0.9628    0.9303   5486500
weighted

In [19]:
EPOCHS = 50
model = AdvancedGAT(num_node_features=len(feature_cols), num_classes=13) # Lấy động features từ mảng cột DataFrame
model = model.to(device)

# 2. Khởi tạo lại Optimizer (Bắt buộc phải tạo lại để nó theo dõi trọng số của GAT)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)
# (Hàm criterion NLLLoss với class_weights của bạn vẫn giữ nguyên không đổi)

# 3. Reset Early Stopping (Nên đổi tên file lưu để không ghi đè mất file GCN cũ)
early_stopping = EarlyStopping(patience=10, path='best_advanced_gat.pt')


print("Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...")
for epoch in range(1, EPOCHS + 1):
    # 1. Chạy huấn luyện và lấy kết quả
    train_loss, train_f1 = train_epoch(epoch, EPOCHS)
    
    # 2. Chạy validation đánh giá
    valid_loss, valid_f1 = valid_epoch(epoch, EPOCHS)
    
    # In kết quả của epoch hiện tại
    print(f"Epoch: {epoch:03d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Macro F1: {train_f1:.4f} | "
          f"Valid Loss: {valid_loss:.4f} - Macro F1: {valid_f1:.4f}")
    
    # 3. Kiểm tra điều kiện dừng sớm
    early_stopping(valid_f1, model)
    scheduler.step(valid_f1) # Điều chỉnh learning rate dựa trên Macro F1 của Validation
    if early_stopping.early_stop:
        print(f"\n🛑 [DỪNG SỚM] Mô hình không cải thiện sau {early_stopping.patience} Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch {epoch}!")
        break
# 1. Nạp lại trọng số tối ưu nhất đã được Early Stopping lưu lại trước đó
model.load_state_dict(torch.load('best_advanced_gat.pt'))
model = model.to(device)
print("Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!")

# 2. Chạy Benchmark chính thức trên tập Test (Chỉ lấy node mới, không tính trùng overlap)
test_f1, test_preds, test_labels = test_benchmark_new_nodes(
    loader=test_loader, 
    window_size=2000, 
    stride=500
)

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...


Epoch 001/50 [Train]:   0%|                                                 | 0/275 [00:00<?, ?it/s]

Epoch: 001/50 | Train Loss: 0.6374 - Macro F1: 0.6317 | Valid Loss: 0.2227 - Macro F1: 0.7124
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 002/50 | Train Loss: 0.2523 - Macro F1: 0.7533 | Valid Loss: 0.1850 - Macro F1: 0.7511
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


...

Epoch: 018/50 | Train Loss: 0.0159 - Macro F1: 0.9097 | Valid Loss: 0.0127 - Macro F1: 0.9094
 -> [EarlyStopping] Bộ đếm tăng: 9 / 10


C:\Users\Admin\AppData\Local\Temp\ipykernel_17652\3992018802.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_advanced_gat.pt'))


Epoch: 019/50 | Train Loss: 0.0151 - Macro F1: 0.9142 | Valid Loss: 0.0130 - Macro F1: 0.9092
 -> [EarlyStopping] Bộ đếm tăng: 10 / 10

🛑 [DỪNG SỚM] Mô hình không cải thiện sau 10 Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch 19!
Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!
Đang chạy Benchmark trên tập Test (Chiến lược: Chỉ đánh giá luồng Mới)...


Test Benchmark: 100%|█████████████████████████████████████████████| 275/275 [00:22<00:00, 12.48it/s]


Tổng số flow thực tế được đánh giá: 5486500
🏆 CHÍNH THỨC - TEST BENCHMARK MACRO F1: 0.9382

📊 Classification Report Chi Tiết:
              precision    recall  f1-score   support

           0     0.9993    0.8984    0.9461    410000
           1     0.9539    0.9997    0.9763   1021936
           2     0.8924    0.8685    0.8803    153398
           3     1.0000    0.9986    0.9993    920554
           4     0.9971    0.9818    0.9894    458988
           5     1.0000    0.9968    0.9984    390000
           6     1.0000    1.0000    1.0000   1149290
           7     0.9970    0.8629    0.9251    115104
           8     0.9969    1.0000    0.9984    262198
           9     0.9841    0.8131    0.8905      3424
          10     0.6807    1.0000    0.8100      1360
          11     0.6569    0.9686    0.7828     43362
          12     0.9999    1.0000    1.0000    556886

    accuracy                         0.9834   5486500
   macro avg     0.9352    0.9529    0.9382   5486500
weighted